In [1]:
# === SESSION BOOTSTRAP ===
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT="/content/drive/MyDrive/UAV_TRUST_Research"; REPO=f"{PARENT}/uav-trust-research"
for fn in (".gitconfig",".git-credentials"):
    p=os.path.join(PARENT,fn)
    if os.path.exists(p): subprocess.run(f'cp "{p}" /root/{fn}',shell=True)
subprocess.run("git config --global credential.helper store",shell=True)
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0,REPO) if REPO not in sys.path else None; print("cwd:",os.getcwd())
else: print("run 00_setup first")

Mounted at /content/drive
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost scikit-learn pandas numpy --quiet

In [3]:
# MATCHED-RATIO UAV-CAS conformal experiment: force identical normal:attack ratios in the
# conformal-calibration set and the in-distribution test set (and match the shifted set to the same ratio),
# so coverage change is attributable to attack-family shift, not to a calibration/test mixture mismatch.
CAS={"file":"data/uav_cas/UAV-CAS_stat.csv","label_col":"Label","normal_value":"Benign",
     "single":{"benign","dos","ddos","blackhole","wormhole","replay"},
     "drops":["config_idx","num_drones","num_bs","payload","pathloss","modulation","mission","tx_power","noise","src_ip","dst_ip"]}
CFG={"seeds":list(range(10)),"alpha":0.10,
     "normal_fracs":{"train":0.60,"cal":0.20,"test_seen":0.10,"test_shift":0.10},
     "family_fracs":{"train":0.60,"cal":0.20,"test_seen":0.20},
     "target_attack_fracs":[0.50, None],   # 0.50 = balanced (primary); None = match to natural calibration mixture
     "xgb":{"n_estimators":300,"max_depth":6,"learning_rate":0.1,"subsample":0.9,"colsample_bytree":0.9,"tree_method":"hist"},
     "report_dir":"reports"}
import os; os.makedirs(CFG["report_dir"],exist_ok=True); print("configured")

configured


In [4]:
import numpy as np, pandas as pd, gc
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score
from src.trust import conformal_qhat

raw=pd.read_csv(CAS["file"],low_memory=False); lab=raw[CAS["label_col"]].astype(str)
keep=lab.str.lower().isin(CAS["single"]) & (~lab.str.contains("+",regex=False)); df=raw[keep].reset_index(drop=True)
NV=[v for v in df[CAS["label_col"]].unique() if str(v).lower()==CAS["normal_value"].lower()][0]
FAMS=[v for v in df[CAS["label_col"]].unique() if v!=NV]
print("rows",len(df),"| normal",repr(NV),"| families",FAMS)

def base_prepare(held_out, seed):
    lc=CAS["label_col"]; nv=NV
    feat=df.drop(columns=[col for col in CAS["drops"] if col in df.columns]+[lc]).apply(pd.to_numeric,errors="coerce")
    feat=feat.drop(columns=[col for col in feat.columns if feat[col].nunique(dropna=False)<=1]).replace([np.inf,-np.inf],np.nan).fillna(0.0)
    X=feat.values.astype(np.float32); y=(df[lc].values!=nv).astype(int); fam=df[lc].values; seen=[f for f in FAMS if f!=held_out]
    def sp(ix,fr,sd):
        ix=np.array(ix); r=np.random.default_rng(sd); r.shuffle(ix); out={}; s=0; ks=list(fr)
        for i,k in enumerate(ks): e=len(ix) if i==len(ks)-1 else s+int(round(fr[k]*len(ix))); out[k]=ix[s:e]; s=e
        return out
    tr=[];ca=[];se=[];sh=[]
    ns=sp(np.where(fam==nv)[0],CFG["normal_fracs"],seed); tr+=list(ns["train"]);ca+=list(ns["cal"]);se+=list(ns["test_seen"]);sh+=list(ns["test_shift"])
    for j,f in enumerate(seen): fs=sp(np.where(fam==f)[0],CFG["family_fracs"],seed+j+1); tr+=list(fs["train"]);ca+=list(fs["cal"]);se+=list(fs["test_seen"])
    sh_attack=list(np.where(fam==held_out)[0])
    idx=dict(tr=np.array(sorted(tr)),ca=np.array(sorted(ca)),se=np.array(sorted(se)),sh_norm=np.array(sorted(sh)),sh_atk=np.array(sh_attack))
    sc=StandardScaler().fit(X[idx["tr"]])
    return X,y,sc,idx
print("base prepare ready")

rows 92131 | normal 'Benign' | families ['DoS', 'DDoS', 'Blackhole', 'Wormhole', 'Replay']
base prepare ready


In [5]:
def match_ratio(idx_norm, idx_atk, target, rng):
    """Return indices resampled so attack fraction == target (or natural if target None -> uses all, returns as-is)."""
    nN,nA=len(idx_norm),len(idx_atk)
    if target is None:  # keep natural composition
        return np.concatenate([idx_norm,idx_atk])
    # want nA/(nA+nN)=target -> choose counts hitting target with max data
    if target<=0 or target>=1: return np.concatenate([idx_norm,idx_atk])
    # attack-limited or normal-limited
    a_for_all_n = int(round(nN*target/(1-target)))   # attacks needed to pair all normals
    if a_for_all_n<=nA: useN,useA=nN, a_for_all_n
    else:
        n_for_all_a=int(round(nA*(1-target)/target)); useN,useA=min(nN,n_for_all_a), nA
    si=rng.choice(idx_norm,min(useN,nN),replace=False); sa=rng.choice(idx_atk,min(useA,nA),replace=False)
    return np.concatenate([si,sa])

def covd(clf,X,idx,y,qh):
    if len(idx)==0: return (np.nan,np.nan,np.nan,0)
    p=clf.predict_proba(X[idx])[:,1]; yy=y[idx]; ia=(1-p)<=qh; ino=p<=qh; inset=np.where(yy==1,ia,ino)
    bal=float(np.mean([inset[yy==k].mean() for k in np.unique(yy)])) if len(np.unique(yy))>1 else float(inset.mean())
    return float(inset.mean()), bal, balanced_accuracy_score(yy,(p>=.5).astype(int)) if len(np.unique(yy))>1 else np.nan, len(idx)

rows=[]
for target in CFG["target_attack_fracs"]:
    tag="balanced_0.5" if target==0.5 else ("natural" if target is None else str(target))
    for F in FAMS:
        for seed in CFG["seeds"]:
            X,y,sc,idx=base_prepare(F,seed); rng=np.random.default_rng(10000+seed)
            # split calibration into prob-cal and conformal-cal; match ratio on the CONFORMAL-cal and ID-test
            ca=idx["ca"]; perm=rng.permutation(len(ca)); h=len(ca)//2; ip,ic=ca[perm[:h]],ca[perm[h:]]
            # conformal-cal matched
            ic_norm=ic[y[ic]==0]; ic_atk=ic[y[ic]==1]
            ic_m=match_ratio(ic_norm,ic_atk,target,rng)
            # ID-test matched (seen test)
            se=idx["se"]; se_norm=se[y[se]==0]; se_atk=se[y[se]==1]
            se_m=match_ratio(se_norm,se_atk,target,rng)
            # shifted test matched: attack = held-out family, normal = sh_norm, matched to same target
            sh_m=match_ratio(idx["sh_norm"], idx["sh_atk"], target, rng)
            Xs=lambda a: sc.transform(X[a]).astype(np.float32)
            clf=xgb.XGBClassifier(objective="binary:logistic",eval_metric="logloss",random_state=seed,**CFG["xgb"]).fit(sc.transform(X[idx["tr"]]).astype(np.float32),y[idx["tr"]])
            # need probs at raw indices; wrap a scaler-aware predict
            class W:
                def __init__(s,clf,sc): s.clf=clf; s.sc=sc
                def predict_proba(s,Xr): return s.clf.predict_proba(s.sc.transform(Xr).astype(np.float32))
            w=W(clf,sc)
            qh=conformal_qhat(w.predict_proba(X[ic_m])[:,1], y[ic_m], alpha=CFG["alpha"])
            id_m=covd(w,X,se_m,y,qh); shc=covd(w,X,sh_m,y,qh)
            rows.append({"target":tag,"held_out":str(F),"seed":seed,"one_minus_q":float(1-qh),
                "cal_attack_frac":float(y[ic_m].mean()),"idtest_attack_frac":float(y[se_m].mean()),
                "id_cov":id_m[0],"shift_cov_marg":shc[0],"shift_cov_bal":shc[1],"shift_balacc":shc[2]})
            del X,y,sc,clf,w; gc.collect()
        print(" ",tag,F,"done")
M=pd.DataFrame(rows); M.to_csv(os.path.join(CFG["report_dir"],"19_uavcas_matched_ratio_raw.csv"),index=False)
print("rows:",len(M))

  balanced_0.5 DoS done
  balanced_0.5 DDoS done
  balanced_0.5 Blackhole done
  balanced_0.5 Wormhole done
  balanced_0.5 Replay done
  natural DoS done
  natural DDoS done
  natural Blackhole done
  natural Wormhole done
  natural Replay done
rows: 100


In [6]:
# Summary: matched-ratio in-distribution coverage (should return toward 0.90) + shift coverage
for tag in M["target"].unique():
    sub=M[M["target"]==tag]
    print(f"\n=== target = {tag} ===")
    print("cal attack frac %.3f | ID-test attack frac %.3f  (now matched)"%(sub["cal_attack_frac"].mean(),sub["idtest_attack_frac"].mean()))
    print("in-distribution marginal coverage: %.3f +/- %.3f  (unmatched baseline was 0.85)"%(sub["id_cov"].mean(),sub["id_cov"].std()))
    g=sub.groupby("held_out").agg(one_minus_q=("one_minus_q","mean"),id_cov=("id_cov","mean"),
        shift_cov_marg=("shift_cov_marg","mean"),shift_cov_bal=("shift_cov_bal","mean"),shift_balacc=("shift_balacc","mean")).round(3)
    print(g.to_string())
M.groupby(["target","held_out"]).agg(id_cov=("id_cov","mean"),shift_cov_marg=("shift_cov_marg","mean"),
    shift_cov_bal=("shift_cov_bal","mean"),one_minus_q=("one_minus_q","mean")).round(3).to_csv(os.path.join(CFG["report_dir"],"19_uavcas_matched_ratio_summary.csv"))


=== target = balanced_0.5 ===
cal attack frac 0.500 | ID-test attack frac 0.500  (now matched)
in-distribution marginal coverage: 0.901 +/- 0.005  (unmatched baseline was 0.85)
           one_minus_q  id_cov  shift_cov_marg  shift_cov_bal  shift_balacc
held_out                                                                   
Blackhole        0.154   0.902           0.806          0.806         0.504
DDoS             0.240   0.901           0.999          0.999         0.984
DoS              0.251   0.901           0.999          0.999         0.986
Replay           0.237   0.902           0.831          0.831         0.504
Wormhole         0.161   0.899           0.780          0.780         0.506

=== target = natural ===
cal attack frac 0.311 | ID-test attack frac 0.475  (now matched)
in-distribution marginal coverage: 0.848 +/- 0.008  (unmatched baseline was 0.85)
           one_minus_q  id_cov  shift_cov_marg  shift_cov_bal  shift_balacc
held_out                                 

In [ ]:
!git add reports/ notebooks/
!git commit -m "19 matched-ratio UAV-CAS conformal: identical normal:attack in conformal-cal, ID-test, and shifted sets; clean in-distribution baseline vs the 0.85 mixture-mismatch result"
!git push origin main